In [ ]:
import sys
import yaml
import numpy as np
import matplotlib.pyplot as plt
from tabpfn import TabPFNRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sys.path.append("..")
from src.data_generation.data_preperation import data_preparation
cfg = yaml.safe_load(open("../config.yaml"))

In [ ]:
train, test = data_preparation(cfg, 20, 50)
X_train, y_train = train[0]
X_test,  y_test  = test[0]

In [ ]:
model = TabPFNRegressor(inference_config={"FINGERPRINT_FEATURE": False})
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
r2   = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.6f}")
print(f"MAE:  {mae:.6f}")
print(f"MAPE: {mape:.4f}%")
print(f"R^2:  {r2:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(y_test, y_pred, s=10, alpha=0.5)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[0].plot(lims, lims, "r--", lw=1)
axes[0].set_xlabel(r"True $\sigma$")
axes[0].set_ylabel(r"Predicted $\sigma$")
axes[0].set_title("Predicted vs True")

residuals = y_pred - y_test
axes[1].scatter(y_test, residuals, s=10, alpha=0.5)
axes[1].axhline(0, color="r", lw=1, linestyle="--")
axes[1].set_xlabel(r"True $\sigma$")
axes[1].set_ylabel("Residual")
axes[1].set_title("Residuals")

plt.tight_layout()
plt.show()


In [ ]:
ttms_g = np.geomspace(cfg["ttm"]["min"], cfg["ttm"]["max"], cfg["ttm"]["n_points"])
ks_g   = np.linspace(cfg["k"]["min"], cfg["k"]["max"], cfg["k"]["n_points"])
K, T = np.meshgrid(ks_g, ttms_g, indexing="ij")

true_grid = np.full(K.shape, np.nan)
pred_grid = np.full(K.shape, np.nan)

k_idx_train = np.searchsorted(ks_g, X_train[:, 0])
t_idx_train = np.searchsorted(ttms_g, X_train[:, 1])
k_idx_test  = np.searchsorted(ks_g, X_test[:, 0])
t_idx_test  = np.searchsorted(ttms_g, X_test[:, 1])

true_grid[k_idx_train, t_idx_train] = y_train
true_grid[k_idx_test,  t_idx_test]  = y_test

pred_grid[k_idx_train, t_idx_train] = y_train   # context points are known, not predicted
pred_grid[k_idx_test,  t_idx_test]  = y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 6), subplot_kw={"projection": "3d"})

axes[0].plot_surface(K, T, true_grid, cmap="viridis", edgecolor="none", alpha=0.9)
axes[0].set_title("True surface")

axes[1].plot_surface(K, T, pred_grid, cmap="viridis", edgecolor="none", alpha=0.9)
axes[1].scatter(X_train[:, 0], X_train[:, 1], y_train, color="red", s=20, zorder=5, label="context points")
axes[1].set_title("Predicted surface (TabPFN)")
axes[1].legend()

for ax in axes:
    ax.set_xlabel("k")
    ax.set_ylabel("tau")
    ax.set_zlabel("sigma")

plt.tight_layout()
plt.show()
